# JUPITER: навчання adaptive RMU та оцінювання після unlearning

Цей навчальний notebook підготовлено **30 жовтня 2026**. Він викликає збережений
тренувальний workflow `evaluation.unlearn_eval.main(...)`, який завантажує дві
копії моделі, виконує adaptive RMU, зберігає модель і оцінює її на BBQ.
Новий запуск цього notebook ще не виконувався.

Жовтневі експерименти виконувалися через CLI на JUPITER: Lapa job 2492541,
Gemma job 2493776, Qwen job 2493909. Їхні фактичні результати відтворює
[виконаний notebook аналізу](../executed/jupiter_rmu_october_2026.ipynb).
Пізніший snapshot коду не доводить точну версію коду цих історичних запусків.

Запускайте тут у GPU-сесії JUPITER з Python/залежностями з
`evaluation/unlearning/jupiter_rmu_20261024/source_snapshot/pyproject.toml`.
Потрібні локальні ваги та оригінальні англомовні StereoSet/BBQ дані у спільному
каталозі команди. Notebook перевіряє їх перед навчанням. Повторні запуски
створюють нові каталоги; збережені історичні моделі та вимірювання не змінюються.

In [ ]:
import hashlib
import importlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

REPO = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "evaluation/unlearning/jupiter_rmu_20261024/manifest.json").is_file()
)
EVIDENCE = REPO / "evaluation/unlearning/jupiter_rmu_20261024"
SOURCE = EVIDENCE / "source_snapshot"
MODEL_ALIAS = "lapa-12b-instruct"
MODEL_ROOT = Path("/e/project1/e-dev-2026d09-075/fairforget/models")
DATA_REPO = Path("/e/project1/e-dev-2026d09-075/fairforget-bias-mitigation/bias-mitigation-kopach1")
RUNS_ROOT = Path("/e/scratch/e-dev-2026d09-075/fairforget-bias-mitigation/runs")
LAYERS = {
    "lapa-12b-instruct": (32, "30,31,32"),
    "gemma3-12b-it": (32, "30,31,32"),
    "qwen3-8b": (17, "15,16,17"),
}
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = RUNS_ROOT / f"notebook-rmu-{run_stamp}-{uuid4().hex[:8]}"
print("Model:", MODEL_ALIAS)
print("New output directory:", RUN_DIR)

## Перевірка джерел, даних, ваг і GPU

Типовий вибір — Lapa; для Gemma/Qwen змініть лише `MODEL_ALIAS` вище.
`loss_positions=words`, beta=1, alpha=5, lr=0.0005, batch=4, seed=42.
Заявлений максимум 257 кроків trainer обмежує числом доступних split-батчів
(у збережених запусках фактично 129).

Forget — стереотипні речення однієї половини English StereoSet gender;
retain — антистереотипні речення іншої. Цей retain не замінює загальну
перевірку корисності моделі. Повна оцінка після навчання — English BBQ
Gender_identity. Це історичний пілот, окремий від поточної панелі Fair-UK.

Наступна клітинка перевіряє frozen hashes та змінює шляхи лише в пам'яті:
вона не редагує `manifest.json`, snapshot конфігурації або вхідні файли.

In [ ]:
manifest = json.loads((EVIDENCE / "manifest.json").read_text())
for entry in manifest["files"]:
    file_path = EVIDENCE / entry["path"]
    if hashlib.sha256(file_path.read_bytes()).hexdigest() != entry["sha256"]:
        raise RuntimeError(f"Frozen evidence changed: {entry['path']}")

sys.path.insert(0, str(SOURCE / "src"))
rmu_config = importlib.import_module("rmu.config")

if not Path(rmu_config.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError("Restart the kernel: another rmu package is already loaded.")
configuration = rmu_config.load_config()
configuration["paths"]["models_root"] = str(MODEL_ROOT)
for key in ("stereoset_files", "bbq_files"):
    configuration["data"][key] = [
        str(DATA_REPO / value) if not Path(value).is_absolute() else value
        for value in configuration["data"][key]
    ]
metadata = Path(configuration["data"]["bbq_metadata"])
configuration["data"]["bbq_metadata"] = str(
    metadata if metadata.is_absolute() else DATA_REPO / metadata
)
MODEL_PATH = Path(rmu_config.resolve_model(MODEL_ALIAS))
INPUT_FILES = [
    *map(Path, configuration["data"]["stereoset_files"]),
    *map(Path, configuration["data"]["bbq_files"]),
    Path(configuration["data"]["bbq_metadata"]),
]


def check_preflight(torch_module):
    if MODEL_ALIAS not in LAYERS:
        raise ValueError("Choose one of the three recorded pilot model aliases.")
    required = [MODEL_PATH / "config.json", MODEL_PATH / "tokenizer_config.json", *INPUT_FILES]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError("Missing local inputs: " + ", ".join(missing))
    weights = list(MODEL_PATH.glob("*.safetensors")) + list(MODEL_PATH.glob("pytorch_model*.bin"))
    if not weights or any(not path.is_file() for path in weights):
        raise FileNotFoundError(f"Local model weights are unavailable: {MODEL_PATH}")
    if not torch_module.cuda.is_available():
        raise RuntimeError("A JUPITER GPU allocation with CUDA is required for training.")
    if RUN_DIR.exists():
        raise FileExistsError(f"Output directory already exists; select a new run: {RUN_DIR}")
    return {str(path): hashlib.sha256(path.read_bytes()).hexdigest() for path in INPUT_FILES}


torch = importlib.import_module("torch")

input_hashes = check_preflight(torch)
print("Local weights, inputs, CUDA and a new output path verified.")
print("BBQ evaluation uses the real in-memory trained model; no mock mode is requested.")

## Параметри нового навчання

Збережена функція `main` виконує `load_model → get_stereoset_data → run_rmu →
run_evaluation`. Вона використовує точні параметри нижче і записує їх разом із
новим результатом. Наступна тренувальна клітинка реально виконує навчання.
Початкові оцінки 23 жовтня залишаються в evidence package; нові треновані ваги
не слід порівнювати з ними як повністю атестовану поточну Fair-UK пару.

In [ ]:
hook_layer, trained_layers = LAYERS[MODEL_ALIAS]
argv = [
    "--model_name_or_path",
    MODEL_ALIAS,
    "--output_dir",
    str(RUN_DIR),
    "--split",
    "--loss_positions",
    "words",
    "--beta",
    "1",
    "--alpha",
    "5",
    "--lr",
    "0.0005",
    "--batch_size",
    "4",
    "--max_num_batches",
    "257",
    "--max_length",
    "512",
    "--seed",
    "42",
    "--layer_id",
    str(hook_layer),
    "--layer_ids",
    trained_layers,
    "--eval_batch_size",
    "8",
    "--save_model",
]
print("python -m evaluation.unlearn_eval", " ".join(argv))

In [ ]:
from evaluation import unlearn_eval as historical_pipeline

if not Path(historical_pipeline.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError("Restart the kernel: a different evaluation package is loaded.")
input_hashes = check_preflight(torch)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(exist_ok=False)
receipt = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_prepared": "2026-10-30",
    "argv": argv,
    "input_sha256": input_hashes,
    "model_path": str(MODEL_PATH),
    "model_config_sha256": hashlib.sha256((MODEL_PATH / "config.json").read_bytes()).hexdigest(),
    "source_snapshot": str(SOURCE),
    "source_manifest_sha256": hashlib.sha256((EVIDENCE / "manifest.json").read_bytes()).hexdigest(),
    "slurm_job_id": os.environ.get("SLURM_JOB_ID"),
    "gpu_training_started": True,
}
(RUN_DIR / "notebook_receipt.json").write_text(json.dumps(receipt, indent=2) + "\n")
historical_pipeline.main(argv)

## Оцінювання та збережені результати нового запуску

`eval_mode` має дорівнювати `model`; mock-результат не приймається.
Перевіряємо також повну кількість BBQ-відповідей та фактичне покриття target mapping.
У жовтневих збережених даних 16 із 5 672 `target_loc` були відсутні (8 на умову).
Первинний scorer включає їхні non-unknown outputs у denominator, але не в
чисельник стереотипних відповідей. Історичні метрики залишено без зміни.
Зниження тренувальних втрат або bias score саме по собі не доводить успішної мітигації.

In [ ]:
import pandas as pd
from IPython.display import display

results = json.loads((RUN_DIR / "bbq/eval_results.json").read_text())
if results.get("eval_mode") != "model":
    raise RuntimeError("The evaluation did not use the real trained model.")
predictions = [
    json.loads(line) for line in (RUN_DIR / "bbq/predictions.jsonl").read_text().splitlines()
]
if len(predictions) != 5672 or results["total_examples"] != len(predictions):
    raise RuntimeError("The full 5,672-item BBQ evaluation is incomplete.")
unmapped = sum(row["target_loc"] is None for row in predictions)
print(f"Target mapping: {len(predictions) - unmapped}/{len(predictions)}")
print("Results:", RUN_DIR)
display(
    pd.DataFrame(
        [
            {
                key: results[key]
                for key in [
                    "total_examples",
                    "overall_accuracy",
                    "ambig_accuracy",
                    "disambig_accuracy",
                    "s_amb",
                    "s_dis",
                ]
            }
        ]
    )
)
losses = pd.read_csv(RUN_DIR / "losses.csv")
print("Actual training steps:", len(losses))
display(losses.head())
losses.set_index("step")[["unlearn_loss", "retain_loss"]].plot(
    title=f"{MODEL_ALIAS}: new adaptive RMU run", logy=True
)

`retain_loss` у первинному журналі вже зважений alpha=5.
Перевірка загальної корисності, інші seeds, довірчі інтервали та повні training-time
вагові/tokenizer/effective-input receipts залишаються окремими вимогами дослідження.
Цей notebook відтворює конкретний історичний метод і не підміняє актуальний
Fair-UK CLI для нового підтверджувального експерименту.